# Инструкция по работе с программным конвейером

Данный блокнот содержит прототип системы автоматического извлечения данных из инженерных чертежей с использованием моделей **YOLOv8s** и **PaddleOCR**. 

## Что делать при ошибках совместимости библиотек?
Среда Google Colab регулярно обновляет встроенные пакеты (например, `numpy` и `PyTorch`). Если при запуске первой ячейки (установка зависимостей) или во время инициализации YOLO вы получаете ошибку совместимости (например, `numpy.dtype size changed`) или ядро "падает", сделайте следующее:
1. В верхнем меню Colab выберите: **Среда выполнения $\rightarrow$ Перезапустить сеанс** (Runtime $\rightarrow$ Restart session).
2. Запустите все заново. Очистка старого кэша памяти решит конфликт C-библиотек.

## Где найти результаты работы и сохраненные файлы?
Все сгенерированные файлы сохраняются в виртуальной файловой системе Colab. Чтобы получить к ним доступ, откройте боковую панель слева и нажмите на значок **папки**.

*   **`/content/dataset/`** — сюда скачивается и распаковывается исходный набор данных (изображения и разметка `labels`).
*   **`/content/runs/detect/runs/train/`** — здесь сохраняются результаты обучения детектора YOLO. Внутри вы найдете:
    *   `weights/best.pt` — лучшие веса обученной модели;
    *   `results.png` — графики метрик обучения (Loss, mAP);
    *   `confusion_matrix.png` — матрицу ошибок;
    *   `val_batch...pred.jpg` — визуальные примеры работы модели с отрисованными рамками.
*   **`/content/result_extraction.json`** — итоговый машиночитаемый файл (результат всего конвейера), который появится в корневой директории после успешного выполнения последней ячейки скрипта. Этот файл можно скачать (Правая кнопка мыши $\rightarrow$ Скачать).

**Как запустить:** Вы можете нажать `Среда выполнения -> Выполнить все` (Runtime -> Run all), скрипт автоматически скачает открытый датасет, проведет конвертацию разметки, обучит модель и выдаст итоговый JSON.

In [ ]:
import sys
import subprocess
import logging

# Настройка логгера с принудительной перезаписью
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True,
    stream=sys.stdout
)

def setup_environment() -> None:
    dependencies = [
        "ultralytics>=8.4.0",
        "paddleocr==2.9.1",
        "paddlepaddle==3.0.0"
    ]

    logging.info("Инициализация проверки системных зависимостей...")

    for package in dependencies:
        logging.info(f"Установка {package}...")
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", package, "-q", "--disable-pip-version-check"]
            )
        except subprocess.CalledProcessError as e:
            logging.error(f"Критическая ошибка при установке пакета {package}: {str(e)}")
            raise SystemExit(1)

    logging.info("Окружение успешно настроено и готово к работе.")

setup_environment()

In [ ]:
import zipfile
import logging
import gdown
from pathlib import Path
import sys

logging.getLogger().addHandler(logging.StreamHandler(sys.stdout))

def download_and_extract_public(file_id: str, extract_dir: str = "/content/dataset") -> None:
    target_dir = Path(extract_dir)
    archive_path = "dataset.zip"

    logging.info("Скачивание датасета...")

    url = f"https://drive.google.com/uc?id={file_id}"

    try:
        gdown.download(url, archive_path, quiet=False)
        logging.info("Файл успешно скачан!")
    except Exception as e:
        logging.error(f"Ошибка скачивания: {str(e)}")
        return

    logging.info("Начало распаковки данных...")
    try:
        target_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive_path, 'r') as zip_ref:
            zip_ref.extractall(target_dir)
        logging.info(f"Распаковка завершена. Данные доступны в: {target_dir.absolute()}")
    except zipfile.BadZipFile:
        logging.error("Скачанный файл поврежден или не является ZIP-архивом.")

FILE_ID = "1JmL3Esa-SxIB4Buipr9cG53C9L0F3wpa"

download_and_extract_public(file_id=FILE_ID)

In [ ]:
import logging
from pathlib import Path
from typing import List

def convert_polygons_to_bbox(directory: str) -> None:
    target_dir = Path(directory)
    if not target_dir.exists():
        logging.warning(f"Директория не найдена: {directory}")
        return

    txt_files = list(target_dir.glob("*.txt"))
    if not txt_files:
        logging.info(f"Файлы разметки отсутствуют в {directory}")
        return

    processed_count = 0
    for file_path in txt_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()

        valid_lines: List[str] = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue

            try:
                class_id = int(parts[0])
                coords = [float(x) for x in parts[1:]]
            except ValueError:
                continue

            if len(coords) == 4:
                valid_lines.append(line)
                continue

            if len(coords) > 4 and len(coords) % 2 == 0:
                xs, ys = coords[0::2], coords[1::2]

                min_x, max_x = max(0.0, min(xs)), min(1.0, max(xs))
                min_y, max_y = max(0.0, min(ys)), min(1.0, max(ys))

                width = max_x - min_x
                height = max_y - min_y

                if width <= 0 or height <= 0:
                    continue

                x_center = min_x + (width / 2.0)
                y_center = min_y + (height / 2.0)
                valid_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

        with open(file_path, 'w', encoding='utf-8') as f:
            f.writelines(valid_lines)

        processed_count += 1

    logging.info(f"Обработка завершена. Обновлено файлов в {directory}: {processed_count}")

convert_polygons_to_bbox('/content/dataset/train/labels')
convert_polygons_to_bbox('/content/dataset/valid/labels')

In [ ]:
import glob
import logging
from ultralytics import YOLO

def train_yolo_model() -> None:
    yaml_files = glob.glob("/content/dataset/**/data.yaml", recursive=True)

    if not yaml_files:
        logging.error("Файл конфигурации data.yaml не найден. Прерывание обучения.")
        return

    data_yaml = yaml_files[0]
    logging.info(f"Инициализация обучения с конфигурацией: {data_yaml}")

    try:
        model = YOLO('yolov8s.pt')
        logging.info("Модель YOLOv8s успешно загружена. Старт обучения.")

        results = model.train(
            data=data_yaml,
            epochs=50,
            imgsz=1024,
            batch=4,
            project='runs/train',
            name='mvp_yolov8s_1024',
            patience=15,
            verbose=False
        )

        save_dir = getattr(results, 'save_dir', 'runs/train/mvp_yolov8s_1024')
        logging.info(f"Обучение завершено. Файлы сохранены в: {save_dir}")

    except Exception as e:
        logging.critical(f"Ошибка при обучении: {str(e)}", exc_info=True)

train_yolo_model()

In [ ]:

import cv2
import json
import os
import glob
import logging
from pathlib import Path
from ultralytics import YOLO
from paddleocr import PaddleOCR

logging.getLogger("ppocr").setLevel(logging.ERROR)

CONFIG = {
    "model_path": '/content/runs/detect/runs/train/mvp_yolov8s_1024/weights/best.pt',
    "padding_px": 6,
    "resize_factor": 2.5,
    "target_class": 'dim_text',
    "img_size": 1024,
    "output_json": '/content/result_extraction.json'
}

def run_inference_pipeline() -> None:
    if not Path(CONFIG["model_path"]).exists():
        logging.error(f"Веса модели не найдены: {CONFIG['model_path']}")
        return

    model = YOLO(CONFIG["model_path"])
    ocr = PaddleOCR(use_angle_cls=True, lang='ru', show_log=False)

    test_images = glob.glob('/content/dataset/valid/images/*.jpg') + glob.glob('/content/dataset/test/images/*.jpg')
    if not test_images:
        logging.error("Тестовые изображения не найдены.")
        return

    test_image_path = test_images[0]
    img = cv2.imread(test_image_path)

    if img is None or img.size == 0:
        logging.error(f"Не удалось декодировать изображение: {test_image_path}")
        return

    h_img, w_img = img.shape[:2]
    logging.info(f"Обработка файла: {os.path.basename(test_image_path)}")

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = model(img_rgb, imgsz=CONFIG["img_size"], verbose=False)[0]

    target_class_id = next((k for k, v in results.names.items() if v == CONFIG["target_class"]), None)

    final_output = {
        "image": os.path.basename(test_image_path),
        "dimensions": []
    }

    if target_class_id is not None and len(results.boxes) > 0:
        for box in results.boxes:
            if int(box.cls[0]) != target_class_id:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

            x1_pad = max(0, x1 - CONFIG["padding_px"])
            y1_pad = max(0, y1 - CONFIG["padding_px"])
            x2_pad = min(w_img, x2 + CONFIG["padding_px"])
            y2_pad = min(h_img, y2 + CONFIG["padding_px"])

            if x1_pad >= x2_pad or y1_pad >= y2_pad:
                continue

            crop = img[y1_pad:y2_pad, x1_pad:x2_pad]
            recognized_text = "UNKNOWN"
            ocr_conf = 0.0

            if crop is not None and crop.shape[0] > 0 and crop.shape[1] > 0:
                try:
                    crop_resized = cv2.resize(crop, None,
                                              fx=CONFIG["resize_factor"],
                                              fy=CONFIG["resize_factor"],
                                              interpolation=cv2.INTER_CUBIC)
                    ocr_result = ocr.ocr(crop_resized, cls=True)

                    if ocr_result and ocr_result[0]:
                        best_line = max(ocr_result[0], key=lambda x: x[1][1])
                        recognized_text = best_line[1][0].strip()
                        ocr_conf = float(best_line[1][1])
                except Exception as e:
                    logging.debug(f"Сбой OCR: {str(e)}")

            final_output["dimensions"].append({
                "text": recognized_text,
                "bbox": [x1, y1, x2, y2],
                "type": "dimension",
                "yolo_confidence": round(float(box.conf[0]), 3),
                "ocr_confidence": round(ocr_conf, 3)
            })

    try:
        with open(CONFIG["output_json"], 'w', encoding='utf-8') as f:
            json.dump(final_output, f, ensure_ascii=False, indent=4)
        logging.info(f"Конвейер успешно завершен. Результаты в {CONFIG['output_json']}")
    except IOError as e:
        logging.error(f"Ошибка записи JSON: {str(e)}")

run_inference_pipeline()

In [ ]:
from ultralytics import YOLO
import glob

data_yaml = glob.glob("/content/dataset/**/data.yaml", recursive=True)[0]
model_nano = YOLO('yolov8n.pt')

model_nano.train(
    data=data_yaml,
    epochs=50,
    imgsz=1024,
    batch=16,
    project='runs/train',
    name='experiment_1_nano_1024'
)